# Method 6 — Two-stage: filter outliers, then inpaint with the Laplacian

**Idea in one paragraph.** The observation is `y = x + noise + a few outliers`.
Outliers are *wrong* values, so instead of trying to average them away we (i) **detect** them by
comparing every node with its neighbours using robust statistics (median / MAD), (ii) **throw them
away**, and (iii) **re-estimate** the thrown-away nodes from the remaining good nodes by
*harmonic interpolation* (each unknown node = weighted average of its neighbours).

Notebook map:
0. Vocabulary (residual, MAD, weighted median, roughness, Laplacian, ...).
1. Load data (Mục IV of the assignment), the shared code block from the group doc, evaluation helpers.
2. The method: Stage 1 (flag, incl. the tested improvements), Stage 2 (refill); code; the two functions for the group (`flag_outliers`, `refill_outliers`).
3. Final `x_hat` on the real data (`k = 2.5`, `lam = 0`), save `xhat_6.npy`, figure.
4. Result row for the group comparison table.
5. Write-up skeleton (Mục VI, 4 sections).

Appendices (optional reading): A. parameter check with `cv_score`; B. synthetic data (sanity check, seeds 0-19); C. Stage 1 variants on all 100 public graphs.

## Vocabulary (read this first)

**Residual.** The leftover gap between what you observed and what you expected: `residual = observed − expected`.
In Stage 1 the observed value is `y_i` (what node `i`'s sensor reads) and the expected value is `m_i`
(what its neighbours suggest it should read), so

`r_i = y_i − m_i`

| residual | meaning |
|---|---|
| `r_i ≈ 0` | node agrees with its neighbours, looks normal |
| `r_i` large, positive | node reads much **higher** than its neighbours |
| `r_i` large, negative | node reads much **lower** than its neighbours |

Large in either direction is suspicious, which is why the flagging rule uses `|r_i|`.
Example (neighbours' median `m_5 = 11`): `y_5 = 11.4` gives `r_5 = 0.4` (normal noise); `y_5 = 30` gives `r_5 = 19` (too big for noise, maybe an outlier).
The word is used the same way everywhere in statistics: in linear regression the residual is the data point minus the fitted line's prediction.
Here the "prediction" is simply the neighbours' median.

**Other terms used in this notebook** (details in section 2):

| term | short meaning |
|---|---|
| node `i`, neighbour `j` | nodes are identified by their node id 0 … n−1; `j` is a node joined to `i` by an edge |
| edge weight `w_ij` | how strongly `i` and `j` are linked (bigger = counts more) |
| `y` / `x` / `x_hat` | observed noisy values / true unknown values / our estimate of `x` |
| outlier | a node whose observation is badly wrong (much more than normal noise) |
| median | middle value after sorting; ignores extreme values |
| weighted median `m_i` | median of the neighbours' `y_j`, where neighbours with bigger `w_ij` count more |
| MAD | median absolute deviation: the median of the distances `|r_i − median(r)|`; a spread measure that outliers barely affect |
| `s` | `1.4826 · MAD`, the MAD converted to the scale of a standard deviation |
| `k` | alarm sensitivity: flag node `i` if `|r_i − median(r)| > k·s`; sets the false-alarm rate (see "The flagging rule and the role of k") |
| Gaussian (normal) data | values follow the bell curve with mean `μ` and standard deviation `σ`: ≈ 68 % within μ ± σ, 95 % within μ ± 2σ, 99.7 % within μ ± 3σ; our assumed model for the small noise |
| roughness / smoothness | how much neighbouring nodes disagree, measured by `xᵀLx = Σ over edges of w_ij (x_i − x_j)²`. Small = neighbours have similar values (smooth signal); large = neighbours differ a lot (rough). Example on a chain: 10, 11, 12, 13, 14 gives 4; the same values shuffled, 10, 14, 11, 13, 12, give 30; a spike 10, 10, 30, 10, 10 gives 800. We assume the true `x` is smooth. Full example below the table |
| kept nodes `U`, flagged nodes `F` | the two groups from Stage 1; a subscript means "only those nodes", e.g. `y_U` = observed values of the kept nodes |
| Laplacian `L` | the matrix `L = D − W` (`L_ii` = total weight of node `i`, `L_ij = −w_ij` for an edge, 0 otherwise). Using it on a signal gives the roughness `xᵀLx = Σ w_ij (x_i − x_j)²` (the *Laplacian quadratic form*). Full example below the table |
| harmonic interpolation | fill a removed node with the weighted average of its neighbours (Stage 2) |
| `mask` | True = node observed; False = node hidden (used by `cv_score`) |
| `cv_score` | cross-validation score: hide 1/5 of the nodes, predict them from the rest, report the median absolute error (lower is better). Full explanation in section 4 |

### Details: roughness and the Laplacian

*Roughness.* `xᵀLx = Σ over every edge (i, j) of w_ij · (x_i − x_j)²` is one number: the squared gap across each edge, weighted, summed.
Small = neighbours agree (smooth); large = neighbours disagree (rough). On a chain `0 — 1 — 2 — 3 — 4` with all weights 1:

| signal | values on nodes 0-4 | gaps along edges | xᵀLx |
|---|---|---|---|
| smooth | 10, 11, 12, 13, 14 | 1, 1, 1, 1 | 4 |
| rough | 10, 14, 11, 13, 12 | 4, 3, 2, 1 | 30 |
| spike | 10, 10, 30, 10, 10 | 0, 20, 20, 0 | 800 |

The first two contain the same values; only their arrangement on the graph differs. One spike makes `xᵀLx` explode.
*How each total is computed* (smooth signal): edges (0,1), (1,2), (2,3), (3,4) have gaps −1, −1, −1, −1; squared 1, 1, 1, 1; total 4.
Shuffled: gaps −4, 3, −2, 1, squared 16, 9, 4, 1, total 30. Spike: gaps 0, −20, 20, 0, squared 0, 400, 400, 0, total 800.
With real weights, multiply each squared gap by its edge weight `w_ij` before adding.

*The Laplacian matrix `L`.* The **Laplacian** is the matrix `L`; the number `xᵀLx` is the **Laplacian quadratic form**
(also called Dirichlet energy): the roughness of `x`, computed with `L`. It is built as `L = D − W`:
- `W` = weight matrix: `W_ij = w_ij` if `i` and `j` are joined by an edge, 0 otherwise;
- `D` = diagonal matrix of each node's total weight, `D_ii = Σ_j w_ij`;
- so `L_ii` = total weight of node `i`, `L_ij = −w_ij` for an edge, 0 otherwise.

For the chain `0 — 1 — 2 — 3 — 4` with all weights 1 (end nodes have 1 edge, middle nodes 2):

```
L = [  1 -1  0  0  0 ]
    [ -1  2 -1  0  0 ]
    [  0 -1  2 -1  0 ]
    [  0  0 -1  2 -1 ]
    [  0  0  0 -1  1 ]
```

Check with the smooth signal `x = (10, 11, 12, 13, 14)`:
`Lx = (10−11, −10+22−12, −11+24−13, −12+26−14, −13+14) = (−1, 0, 0, 0, 1)`, then `xᵀLx = 10·(−1) + 14·1 = 4`,
the same number as the edge-by-edge sum. The edge sum is easier to understand; the matrix is what the code uses
(`nx.laplacian_matrix(...)` in section 1, `L_FF` and `L_FU` in Stage 2, Step 4).
Note: row `i` of `Lx` equals `Σ_j w_ij (x_i − x_j)`, how much node `i` differs from its neighbours. Stage 2, Step 3 sets exactly this to 0 for flagged nodes.

## 1. Data and shared code

In [ ]:
# ---- Mục III/IV of the assignment: load G, y, positions ---------------------
# NOTE: replace this placeholder with the REAL representative MSSV of your group
# (same one used in the group notebook), otherwise you get a different graph.
import json, hashlib
from pathlib import Path
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

MSSV_DAI_DIEN = "20260001"          # <-- change to the group's real representative MSSV

def assign_mssv_to_case(student_id, cases):
    # Same deterministic MSSV -> case mapping as in the assignment notebook.
    sid = str(student_id).strip()
    if sid in cases:
        return sid
    case_ids = sorted(cases, key=lambda x: (0, int(x)) if x.isdecimal() else (1, x))
    k = len(case_ids)
    if sid.isdecimal():
        index = int(sid) % k
    else:
        index = int.from_bytes(hashlib.sha256(sid.encode("utf-8")).digest(), "big") % k
    return case_ids[index]

def get_problem(student_id, file_path="public_cases.json"):
    db = json.loads(Path(file_path).read_text(encoding="utf-8"))
    case = db["cases"][assign_mssv_to_case(student_id, db["cases"])]
    G = nx.Graph()
    for i, xy in enumerate(case["positions"]):
        G.add_node(i, pos=tuple(map(float, xy)))
    for u, v, w in case["edges"]:
        G.add_edge(int(u), int(v), weight=float(w))
    y = np.asarray(case["y"], dtype=float)
    assert nx.is_connected(G) and len(y) == G.number_of_nodes()
    return G, y

G, y = get_problem(MSSV_DAI_DIEN)
positions = nx.get_node_attributes(G, "pos")
n = len(y)
print("n nodes =", n, "| n edges =", G.number_of_edges())

In [ ]:
# ---- Shared "contract" block copied from the group doc ----------------------
# L = weighted graph Laplacian = D - W.  For any signal x:  x^T L x = sum_{(i,j)} w_ij (x_i - x_j)^2
# i.e. x^T L x is SMALL when neighbouring nodes have similar values (a "smooth" signal).
n = len(y)
L = nx.laplacian_matrix(G, nodelist=range(n), weight="weight").toarray()

# (a) Synthetic data with a KNOWN x, so we can measure RMSE.
def make_synthetic(G, y, seed):
    rng = np.random.default_rng(seed)
    lam, U = np.linalg.eigh(L)
    # smooth ground truth = random combination of the 5 lowest-frequency eigenvectors
    x = U[:, :5] @ (rng.normal(size=5) / (1 + lam[:5]))
    # rescale so x has the same centre/spread as the real y
    x = np.median(y) + np.std(y) * x / np.std(x)
    sigma = 0.1 * np.std(y)
    y_syn = x + rng.normal(0, sigma, n)               # Gaussian noise
    idx = rng.choice(n, 3, replace=False)             # 3 outlier nodes
    y_syn[idx] += rng.choice([-1, 1], 3) * rng.uniform(4, 8, 3) * sigma
    return x, y_syn

def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))

# (b) Parameter selection on REAL data: hide 1/5 of the nodes, predict them again.
def cv_score(solve, y, params, k=5, seed=0):
    folds = np.random.default_rng(seed).permutation(n) % k
    err = []
    for f in range(k):
        mask = folds != f                              # True = observed
        x = solve(G, y, mask=mask, **params)
        err.append(np.abs(x[~mask] - y[~mask]))
    return np.median(np.concatenate(err))              # median => robust to outliers in y

In [ ]:
# ---- Evaluation helpers (used in section 4 and the appendices) ---------------
def cv_mean(params, seeds=range(5)):
    # cv_score averaged over 5 different random splits, so one lucky/unlucky split does not decide. Lower = better.
    return np.mean([cv_score(solve_two_stage, y, params, seed=sd) for sd in seeds])

def synth_rmse(params, seeds=range(20)):
    # RMSE against the KNOWN x on synthetic data, one value per seed (seeds 0-19, as agreed in the group doc).
    out = []
    for sd in seeds:
        x_t, y_s = make_synthetic(G, y, sd)
        out.append(rmse(solve_two_stage(G, y_s, **params), x_t))
    return np.array(out)

## 2. The method

### Stage 1 — detect outliers (robust, neighbour-based)

**Goal:** for every node `i`, guess what value it *should* have by looking only at its neighbours,
then compare with what it actually reads. That guess is `m_i`.

**How to compute `m_i` (weighted median of the neighbours)**

Notation: nodes are identified by their **node id** (0, 1, ..., n-1), as in the data. `i` is the node we judge, `j` is one of its neighbours.
- **Step 1, neighbours:** the nodes `j` joined to `i` by an edge. Each edge has a weight `w_ij` (bigger = that neighbour counts more).
- **Step 2, collect pairs:** for every neighbour `j` write down `(y_j, w_ij)` = its observed value and its edge weight.
- **Step 3, sort** the pairs from the smallest `y_j` to the largest.
- **Step 4, walk down the sorted list** adding up the weights (running total).
  Stop at the first neighbour where the running total reaches **half of the total weight**.
  That neighbour's `y_j` is `m_i`.

**Worked example (made-up numbers).** Take node `i = 5`. Suppose its neighbours are nodes `j = 3, 8, 12, 20`.
After Step 3 they are sorted by value:

| neighbour node j | y_j | w_5j | running total of weights |
|---|---|---|---|
| 8  | 10 | 1 | 1 |
| 3  | 11 | 3 | 4 |
| 20 | 12 | 1 | 5 |
| 12 | 50 | 1 | 6 |

Total weight = 6, half = 3. The running total first reaches 3 at node 3 (it jumps 1 → 4), so `m_5 = y_3 = 11`.
The odd value 50 (node 12) has no influence. A plain average would give (10 + 3·11 + 12 + 50)/6 = 17.5, dragged up by node 12. That is why we use a *median*.
(With equal weights this is just the ordinary median.)

**Residual.** `r_i = y_i - m_i`. If `y_5 = 11.4` then `r_5 = 0.4` (normal). If `y_5 = 30` then `r_5 = 19` (suspicious).
If node `i` is a normal node, `r_i` is small (signal is smooth + small noise). If it is an outlier, `|r_i|` is large.
The scale `s` below tells us how large "large" is.

**What `s` is.** `s` estimates the *typical size of the residuals at clean nodes* (how far a normal node usually sits from its neighbours' median).
We could use the standard deviation of `r`, but the outliers we are hunting would inflate it, raising the threshold enough to hide themselves.
So we use the **MAD** (median absolute deviation), which the outliers barely affect:

- `c = median(r)` (typical residual, usually close to 0)
- `d_i = |r_i − c|` (distance of each residual from it)
- `MAD = median(d_i)` (typical distance)

Example: `r = [1, 2, 3, 4, 100]` gives `c = 3`, `d = [2, 1, 0, 1, 97]`, so `MAD = 1`. The standard deviation would be about 43, blown up by the single value 100.

**Where 1.4826 comes from.** MAD and the standard deviation σ measure spread on different scales. For Gaussian data, exactly half the points lie
within ±0.6745·σ of the centre (0.6745 = `Φ⁻¹(0.75)`, the 75th percentile of the standard normal), so `MAD ≈ 0.6745·σ` and therefore

`s = MAD / 0.6745 = 1.4826 · MAD`

It is just a unit conversion that makes `s` comparable to a standard deviation (exact for Gaussian data, approximate otherwise).

### The flagging rule and the role of `k`

**Why flag at all?** Stage 2 needs to know *which readings `y_i` to trust*. Trusted nodes keep their value; flagged nodes are thrown away
and refilled from their neighbours. Flagging is that yes/no decision, made separately for every node.

**The question asked for each node:** "Is this node's gap from its neighbours (`|r_i|`) too big to be explained by normal noise?"
- `s` = the size of a *typical* gap at a clean node (its unit of measure).
- `k` = how many typical gaps count as "too big".

Rule: flag node `i` (as a *suspected* outlier) if `|r_i − median(r)| > k · s`. Think of `k` as the **sensitivity of an alarm**:
a small `k` rings at small gaps, a large `k` rings only at huge gaps.

**Worked example (made-up numbers).** Suppose `s = 0.5`, `median(r) = 0`, and six nodes have these residuals:

| node | r_i | flagged at k = 1 (limit 0.5) | k = 2 (limit 1.0) | k = 3 (limit 1.5) |
|---|---|---|---|---|
| 2  | 0.1  | no  | no  | no  |
| 7  | −0.3 | no  | no  | no  |
| 11 | 0.6  | **yes** | no  | no  |
| 15 | −0.8 | **yes** | no  | no  |
| 21 | 1.4  | **yes** | **yes** | no  |
| 28 | 4.0  | **yes** | **yes** | **yes** |

Node 28 (8 typical gaps away) is almost surely an outlier and is caught at every `k`.
Nodes 11 and 15 are only 1-2 typical gaps away, which normal noise does all the time; with `k = 1` they are flagged *wrongly*.

**What `k` controls: the false-alarm rate.** If the noise is Gaussian, this is how often a *clean* node is flagged by accident:

| k | chance a clean node exceeds `k·s` | expected false flags among 32 nodes |
|---|---|---|
| 1.0 | ≈ 31.7 % | ≈ 10 |
| 1.5 | ≈ 13.4 % | ≈ 4.3 |
| 2.0 | ≈ 4.6 % | ≈ 1.5 |
| 2.5 | ≈ 1.2 % | ≈ 0.4 |
| 3.0 | ≈ 0.27 % | ≈ 0.09 |

`k` does **not** say "find the top 1 % of nodes". The number of real outliers is a separate, unknown quantity
(the synthetic generator plants 3 of 32, about 9 %). `k` only sets how often good nodes get flagged by mistake.

**Why not a small `k` such as 1?** About 10 good nodes would be flagged. Each wrong flag (a) throws away a good measurement and replaces it
with a neighbour average, and (b) leaves fewer trusted neighbours for the other flagged nodes, so the refill in Stage 2 gets worse.

**Why not a huge `k`?** Real outliers slip through and their wrong values pull the estimate. (`k = 1e9` in section 4 means "never flag", the no-filter baseline.)

**So `k` is a trade-off**, and the usual sweet spot is about 2.5-3: almost no false alarms, but outliers a few `s` away are still caught.
We do not pick it by reasoning alone: section 4 tries several `k` and keeps the one with the lowest `cv_score`.

**Caveat.** A flag is a bet, not a proof: a real sharp change in the signal (e.g. a boundary between two regions) also gives a large residual.

**Is this rule standard?**
- *Standard:* "flag if distance from the median > k · 1.4826 · MAD" is a classical robust-statistics rule (the MAD rule / Hampel identifier, after Hampel's 1974 work on robust estimation).
  Leys et al. (2013, *J. Experimental Social Psychology*) recommend it over mean ± SD with k = 2.5 (moderately conservative) or 3 (very conservative).
  Rousseeuw & Croux (1993, *JASA*) discuss the MAD and its consistency constant.
- *Known idea:* comparing each point with the median of its neighbours is the Hampel filter for time series (sliding window).
- *Our adaptation:* the window is replaced by the **graph neighbourhood**, with a weighted median using the edge weights.
- *Caveat:* `m_i` is itself computed from noisy values and neighbouring residuals are correlated, so the false-flag rates above are approximate.

### Improving Stage 1: "victims" and the threshold (tested)

**Problem 1: victims.** The neighbour median only resists outliers when they are a *minority* of the neighbours.
Nodes here have only 4-8 neighbours, so 2 outliers next to a clean node can drag its median, and the clean node looks abnormal and gets flagged.
Example (case 20260018): nodes 28 (17.70) and 18 (18.65) are low; their neighbours 0, 14, 15 read about 20.1-20.5, which is normal,
but 2 of their 4 neighbours are the low nodes, so their medians drop to 18.65 and all five nodes get flagged.

**Fix tried: iterative re-judging** (`iterative=True`). Flag only the single worst node, recompute every neighbour median *without* it,
and repeat until nothing is above the threshold. `s` is estimated once on the first pass and then kept fixed.
- *Full version* (`iterative=True`): may also flag nodes that were not flagged in the first pass.
- *Prune version* (`iterative="prune"`): only re-judges the first-pass flags, so it can **remove victims but never adds new flags**.

**Problem 2: tight threshold in very smooth cases.** In case 20260018 `s = 0.39`, the smallest of all cases, so the threshold `2.5·s ≈ 1.0` is tight.

**Fix tried: a floor on `s`** (`s_floor`). Require `s ≥ s_floor × robust spread of y`. We set `s_floor = 0.49`, the 10th percentile of
`s / spread(y)` over the 100 public cases.

**Results** (Appendix C reproduces them; `k = 2.5`, `lam = 0`):

| Stage 1 variant | synthetic RMSE (100 graphs × 5 seeds) | planted outliers caught | false flags per run | real data: cases with adjacent flags | real data: mean `cv_score` |
|---|---|---|---|---|---|
| no processing (`x_hat = y`) | 0.2956 | – | – | – | – |
| original | 0.2471 | 52 % | 1.26 | 10 | 0.6507 |
| **prune** (`iterative="prune"`) | **0.2443** | 49 % | **1.05** | **7** | 0.6515 |
| full iterative | 0.2759 | 51 % | 1.74 | 10 | 0.6518 |
| floor on `s` | 0.2907 | 9 % | 0.19 | 8 | 0.6511 |
| full iterative + floor | 0.2980 | 9 % | 0.25 | 8 | 0.6521 |

**What we keep.**
- **Prune** is the only variant that helps: fewer false flags, slightly lower synthetic RMSE, fewer adjacent-flag groups, same `cv_score` (differences of 0.001 are noise).
  It removes exactly the victims found by eye: nodes 0, 14, 15 (case 20260018), 18 (case 20260028), 29 (case 20260014), 25 (case 20260056).
  **The main result uses `iterative="prune"`.**
- **Full iterative** adds new false flags: once a node is removed, its neighbours' medians use fewer values and become noisier.
- **Floor on `s`** almost stops detection on synthetic data (9 % caught). The synthetic noise (0.1 × std(y)) is much smaller than the residual
  spread of the real cases, so a floor calibrated on the real cases is far too high there. It is kept as an option (`s_floor`) but not used.
- Even the best variant catches only about half of the planted outliers. The main reason: the residual `y_i − m_i` contains the real change of the
  smooth signal between neighbours, not only noise, so `s` is larger than the noise and outliers of 4-8 noise-σ often stay under `2.5·s`.
  This is a limit of the method to state in the write-up.

### Stage 2 — refill the flagged nodes (harmonic interpolation)

**Notation.**
- `y` = what the sensors observed (given). `x` = the true values (unknown). `x_hat` = our estimate, what we submit.
  Below, "`x`" inside the formulas means the estimate being built.
- `U` = kept (trusted) nodes, `F` = flagged nodes, both from Stage 1. A subscript means "only those nodes":
  if the kept nodes are 0, 1, 2, 4 then `y_U = (y_0, y_1, y_2, y_4)`.

**Step 1: keep the trusted readings.** `x_U = y_U`: for every kept node the observed value *is* the prediction.
Only flagged nodes get new values. Example with node 3 flagged:

| node | observed y | kept / flagged | x_hat |
|---|---|---|---|
| 0 | 10.2 | kept | 10.2 (copied) |
| 1 | 9.8 | kept | 9.8 (copied) |
| 2 | 10.5 | kept | 10.5 (copied) |
| 3 | 30.0 | **flagged** | **recomputed from its neighbours** |
| 4 | 10.1 | kept | 10.1 (copied) |

**Step 2: choose the flagged values that make the signal as smooth as possible.**

We want the signal to be as smooth as possible, i.e. its **roughness** `xᵀLx` as small as possible (see "roughness / smoothness" and "Laplacian" in the Vocabulary, with a full example below the table).

*Minimise over the flagged nodes only.* Chain `1 — 4 — 9`, weights 1. Node 4 is flagged; nodes 1 (`y = 10`) and 9 (`y = 16`) are kept and fixed.
Then `xᵀLx = (10 − x_4)² + (x_4 − 16)²`:

| x_4 | (10 − x_4)² | (x_4 − 16)² | xᵀLx |
|---|---|---|---|
| 10 | 0 | 36 | 36 |
| 12 | 4 | 16 | 20 |
| **13** | 9 | 9 | **18 (smallest)** |
| 14 | 16 | 4 | 20 |
| 16 | 36 | 0 | 36 |

The smoothest choice is the average of the neighbours, 13.

**Step 3: the formula.** Set the derivative of `xᵀLx` with respect to a flagged `x_i` to zero:
`2 Σ_j w_ij (x_i − x_j) = 0`, so

`x_i = Σ_j w_ij x_j / Σ_j w_ij`

**Each flagged node = weighted average of its neighbours** (closer neighbours, bigger `w_ij`, pull harder).
A signal with this property is called *harmonic*, hence "harmonic interpolation".

*Example 1, one flagged node.* Node 5 is flagged; its kept neighbours are node 3 (`y = 11`, `w = 3`), node 8 (`y = 10`, `w = 1`), node 20 (`y = 12`, `w = 1`):
`x_5 = (3·11 + 1·10 + 1·12) / (3 + 1 + 1) = 55 / 5 = 11`.

*Example 2, two flagged nodes next to each other.* Chain `1 — 4 — 5 — 9`, weights 1; nodes 4 and 5 flagged, `y_1 = 10`, `y_9 = 16` kept.
Each average contains the other unknown:
`x_4 = (10 + x_5) / 2` and `x_5 = (x_4 + 16) / 2`  ⇒  `x_4 = 12`, `x_5 = 14`. The chain reads 10, 12, 14, 16: a straight line between the trusted ends.

**Step 4: matrix form (what the code solves).** The condition "`Σ_j w_ij (x_i − x_j) = 0` for every flagged `i`" is exactly
"row `i` of `L x` is 0 for every `i` in `F`". Split `x` into its flagged and kept parts:

`L_FF x_F + L_FU x_U = 0`  with `x_U = y_U`  ⇒  **`L_FF x_F = − L_FU y_U`**

- `L_FF` = rows and columns of `L` for flagged nodes (how flagged nodes link to each other).
- `L_FU` = rows for flagged nodes, columns for kept nodes (how flagged nodes link to kept ones).

In Example 2 (order: nodes 4, 5): `L_FF = [[2, −1], [−1, 2]]`, `−L_FU y_U = [10, 16]`, and
`[[2, −1], [−1, 2]] · [x_4, x_5] = [10, 16]` gives `x_4 = 12`, `x_5 = 14`. One `np.linalg.solve` does all the averages at once.

**When does it work?** The system has exactly one solution when every group of flagged nodes touches at least one kept node.
Our graph is connected, so this holds as long as at least one node is kept.

**Properties (for the write-up).**
- No new peaks or dips: a refilled value always lies between its neighbours' values. Good when the outlier was a spike; bad if the true signal really has a peak there.
- Kept nodes are untouched, so their small Gaussian noise remains (see the extension below).
- Closed form: one linear solve, no iterations, no convergence check needed.

### Optional extension (`lam > 0`)
The pure method keeps `x_U = y_U`, so the noise on kept nodes stays. The extension instead minimises

`Σ_{i in U} (x_i − y_i)² + lam · xᵀLx`

- first term: stay close to the trusted observations (flagged nodes are not in it);
- second term: be smooth; `lam` sets how much smoothness matters.

Setting the gradient to zero gives `(M + lam·L) x = M y`, with `M` = diagonal matrix with 1 for kept nodes and 0 for flagged ones.
Small `lam` ≈ the pure method; large `lam` flattens everything toward one constant. `lam = 0` is exactly the method in the group doc;
`lam > 0` is an add-on we compare in section 4.

In [ ]:
def weighted_median(values, weights):
    # Computes m_i from a node's neighbours. `values` = y_j of the neighbours, `weights` = w_ij.
    # Example (worked example above): values=[10,11,12,50], weights=[1,3,1,1]  ->  returns 11.
    order = np.argsort(values)                         # Step 3: sort neighbours by value (small -> large)
    v, w = np.asarray(values)[order], np.asarray(weights)[order]
    c = np.cumsum(w)                                   # Step 4: running total of weights, e.g. [1, 4, 5, 6]
    half = 0.5 * c[-1]                                 #         half of the total weight, e.g. 3
    return v[np.searchsorted(c, half)]                 #         first value whose running total reaches `half`

def robust_spread(v):
    # 1.4826 * MAD: a standard-deviation-like spread that outliers barely affect.
    v = np.asarray(v, dtype=float)
    return 1.4826 * np.median(np.abs(v - np.median(v)))

def detect_outliers(G, y, mask=None, k=2.5, s_min=0.0, s_fixed=None):
    # Stage 1 (one pass). Returns (flagged boolean array, residuals r, robust scale s).
    # `mask` (True = observed) lets cv_score hide nodes: hidden nodes are NOT used as neighbours
    # and are never flagged (they are "unknown", handled in stage 2).
    # `s_min`  : floor for s (improvement 2). `s_fixed`: use this s instead of estimating it (used by the iterative version).
    n = len(y)
    obs = np.ones(n, dtype=bool) if mask is None else np.asarray(mask, dtype=bool)

    # m[i] = "what node i should read, judged only by its neighbours" (weighted median, see section 2)
    m = np.full(n, np.nan)
    for i in range(n):
        nb = [j for j in G.neighbors(i) if obs[j]]     # Step 1: node ids j of i's neighbours (observed only)
        if nb:                                         # a node whose neighbours are all hidden gets no m_i
            vals = y[nb]                               # Step 2: neighbours' observed values y_j ...
            wts = [G[i][j]["weight"] for j in nb]      #         ... and edge weights w_ij
            m[i] = weighted_median(vals, wts)          # Steps 3-4: sort + walk to half the total weight

    r = y - m                                          # residual vs neighbourhood
    valid = obs & ~np.isnan(r)                         # nodes we can actually judge
    r0 = np.median(r[valid])                           # centre of residuals (usually ~0)
    if s_fixed is None:
        s = robust_spread(r[valid])                    # robust sigma (1.4826 * MAD)
        s = max(s, s_min, 1e-8 * np.std(y))            # floor (improvement 2) + guard against s = 0
    else:
        s = s_fixed
    flagged = valid & (np.abs(r - r0) > k * s)
    return flagged, r, s

def detect_outliers_iterative(G, y, mask=None, k=2.5, s_min=0.0, only_first_pass=False):
    # Improvement 1: flag ONE node at a time (the worst), then recompute every neighbour median WITHOUT it.
    # A clean node that only looked odd because an outlier sat among its neighbours ("victim") is then judged again
    # with that outlier removed, and usually stops looking odd.
    # s is estimated once on the first pass and then kept fixed, so removing outliers does not tighten the threshold.
    n = len(y)
    obs = np.ones(n, dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    first, r, s = detect_outliers(G, y, mask=obs, k=k, s_min=s_min)  # first pass: s (and the first-pass flags)
    flagged = np.zeros(n, dtype=bool)
    for _ in range(n):
        trusted = obs & ~flagged                       # flagged nodes no longer count as neighbours
        cand, r, _ = detect_outliers(G, y, mask=trusted, k=k, s_fixed=s)
        if only_first_pass:
            cand &= first                              # option: only re-judge first-pass flags (can remove victims, never adds new flags)
        if not cand.any():
            break
        r0 = np.median(r[trusted & ~np.isnan(r)])
        dev = np.where(cand, np.abs(r - r0), -np.inf)
        flagged[np.argmax(dev)] = True                 # flag only the single worst node, then re-judge the rest
    return flagged, r, s

def inpaint(L, y, keep, lam=0.0):
    # Stage 2. `keep` = boolean array of nodes whose y we trust (the set U). Others (the set F) are re-estimated.
    n = len(y)
    if not keep.any():                                 # nothing trusted: fall back to a constant
        return np.full(n, np.median(y))
    F = ~keep
    if not F.any():
        return y.copy()
    if lam == 0.0:
        x = y.copy()                                   # Stage 2, Step 1: x_U = y_U (kept nodes copy their observation)
        L_FF = L[np.ix_(F, F)]                         # Stage 2, Step 4: Laplacian block flagged x flagged
        L_FU = L[np.ix_(F, keep)]                      #         Laplacian block flagged x kept
        x[F] = np.linalg.solve(L_FF, -L_FU @ y[keep])  #         solve L_FF x_F = -L_FU y_U (each x_i = weighted avg of neighbours)
        return x
    # Extension: minimise sum_{i in U} (x_i - y_i)^2 + lam * x^T L x  ->  (M + lam L) x = M y
    M = np.diag(keep.astype(float))                    # 1 for trusted nodes, 0 for flagged ones
    return np.linalg.solve(M + lam * L, M @ y)

def flag_nodes(G, y, mask=None, k=2.5, iterative=False, s_floor=0.0):
    # iterative: False | True ("full") | "prune" (only re-judge first-pass flags)
    # Stage 1 with the chosen options. s_floor is RELATIVE: s >= s_floor * robust spread of the observed y.
    obs = np.ones(len(y), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    s_min = s_floor * robust_spread(y[obs])
    if iterative:
        return detect_outliers_iterative(G, y, mask=obs, k=k, s_min=s_min, only_first_pass=(iterative == "prune"))
    return detect_outliers(G, y, mask=obs, k=k, s_min=s_min)

def solve_two_stage(G, y, mask=None, k=2.5, lam=0.0, iterative="prune", s_floor=0.0):
    # Contract of the group doc: solve_<name>(G, y, mask=None, **params) -> ndarray of length n.
    # Defaults = the main result (k=2.5, lam=0, iterative="prune"). iterative=False is the original Stage 1; True and s_floor > 0 are the other
    # Stage 1 improvements tested in section 2 ("Improving Stage 1") and Appendix C.
    y = np.asarray(y, dtype=float)
    obs = np.ones(len(y), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    flagged, _, _ = flag_nodes(G, y, mask=obs, k=k, iterative=iterative, s_floor=s_floor)
    keep = obs & ~flagged                              # trusted = observed AND not an outlier
    L_G = nx.laplacian_matrix(G, nodelist=range(len(y)), weight="weight").toarray()   # Laplacian of THIS graph
    return inpaint(L_G, y, keep, lam=lam)

### Functions for the group: `flag_outliers` and `refill_outliers`

Two functions so that other methods can reuse the two stages separately. Same code as `solve_two_stage`
(the check below confirms they give identical results). They are also saved as the module `method6_outliers.py`
(`from method6_outliers import flag_outliers, refill_outliers`).

**`flag_outliers(G, y, k=2.5, mask=None, prune=True, return_details=False)`** (Stage 1)

| input | meaning |
|---|---|
| `G` | networkx graph, nodes `0 … n−1`, every edge has a `"weight"` (bigger = closer sensors) |
| `y` | observed values, length `n`, `y[i]` belongs to node `i` |
| `k` | threshold in typical gaps `s` (default 2.5; larger = fewer flags) |
| `mask` | optional bool array, True = observed, False = hidden/missing (not used as neighbour, never flagged) |
| `prune` | True = un-flag "victims" (see "Improving Stage 1") |
| `return_details` | True = also return the numbers behind the decision |

Output: `flagged`, a bool array of length `n`, **True = suspected outlier**. With `return_details=True` also a dict:
`neighbour_median` (`m_i`), `residual` (`r_i`), `z`, `s` (first pass, before pruning).

**`refill_outliers(G, y, flagged, lam=0.0)`** (Stage 2)

| input | meaning |
|---|---|
| `G`, `y` | as above (values of flagged nodes are ignored, they may be NaN) |
| `flagged` | bool array, True = do not trust `y[i]` (outliers, and also any missing node) |
| `lam` | 0 = only flagged nodes change (weighted average of neighbours); > 0 = also smooth the trusted nodes |

Output: `x_hat`, a float array of length `n`: `x_hat[i] = y[i]` for trusted nodes (`lam = 0`), the new value for flagged nodes.

**How others can use them**
- *Flags only* (recommended): pass them as a mask to your own solver, so your model estimates those nodes itself:
  `x = solve_<yours>(G, y, mask=~flag_outliers(G, y))`. When you compute `cv_score`, call `flag_outliers` inside every fold
  (with that fold's `mask`), otherwise the hidden nodes have already influenced the flags and the score looks too good.
- *Flags + new values*: `x = refill_outliers(G, y, flag_outliers(G, y))`, then use `x` as the input of your method.
  Refilled values are averages of neighbours, not measurements: your method will treat them as real data.

In [ ]:
def flag_outliers(G, y, k=2.5, mask=None, prune=True, return_details=False):
    """Stage 1 of method 6: decide which nodes are suspected outliers.

    Inputs
    ------
    G : networkx.Graph
        Nodes 0 .. n-1. Every edge has a "weight" (bigger = closer, more similar sensors).
    y : array-like of length n
        Observed values; y[i] belongs to node i.
    k : float, default 2.5
        Threshold, in "typical gaps" s. Larger k = fewer flags.
    mask : bool array of length n, optional
        True = observed, False = hidden/missing. Hidden nodes are not used as neighbours and are never flagged.
        Default: all nodes observed.
    prune : bool, default True
        Re-judge the flagged nodes one at a time and un-flag "victims" (clean nodes that only looked odd
        because outliers sat among their neighbours).
    return_details : bool, default False
        Also return the numbers behind the decision.

    Output
    ------
    flagged : bool array of length n
        True = suspected outlier: do not trust y[i].
    details : dict (only if return_details=True), arrays of length n from the first pass (before pruning):
        "neighbour_median" (m_i), "residual" (r_i = y_i - m_i), "z" ((r_i - median(r)) / s), and "s" (a float).

    Example
    -------
    flagged = flag_outliers(G, y)
    np.flatnonzero(flagged)          # node ids of the suspected outliers
    """
    y = np.asarray(y, dtype=float)
    obs = np.ones(len(y), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    y_safe = np.where(obs, y, np.nanmedian(y[obs]))    # values of hidden nodes are never used; avoids NaN issues
    flagged, _, _ = flag_nodes(G, y_safe, mask=obs, k=k, iterative="prune" if prune else False)
    if not return_details:
        return flagged
    _, r, s = detect_outliers(G, y_safe, mask=obs, k=k)
    valid = obs & ~np.isnan(r)
    z = (r - np.median(r[valid])) / s
    return flagged, {"neighbour_median": y_safe - r, "residual": r, "z": z, "s": s}

def refill_outliers(G, y, flagged, lam=0.0):
    """Stage 2 of method 6: compute new values for the flagged nodes.

    Inputs
    ------
    G : networkx.Graph
        Same graph as for flag_outliers.
    y : array-like of length n
        Observed values (entries of flagged nodes are ignored, they may be NaN).
    flagged : bool array of length n
        True = do not trust y[i]: the outliers from flag_outliers, and also any missing/hidden node.
    lam : float, default 0
        0  -> only flagged nodes change: each becomes the weighted average of its neighbours
              (harmonic interpolation: solve L_FF x_F = -L_FU y_U).
        >0 -> also smooth the trusted nodes: solve (M + lam*L) x = M y.

    Output
    ------
    x_hat : float array of length n
        With lam = 0: x_hat[i] = y[i] for trusted nodes, and the new value for flagged nodes.

    Example
    -------
    flagged = flag_outliers(G, y)
    x_hat = refill_outliers(G, y, flagged)
    {int(i): x_hat[i] for i in np.flatnonzero(flagged)}   # new values of the outliers
    """
    y = np.asarray(y, dtype=float)
    flagged = np.asarray(flagged, dtype=bool)
    y_safe = np.where(flagged, 0.0, y)                 # untrusted values never enter the result
    L_G = nx.laplacian_matrix(G, nodelist=range(len(y)), weight="weight").toarray()
    return inpaint(L_G, y_safe, ~flagged, lam=lam)

In [ ]:
# Example: flag, inspect the numbers, refill. Then check it matches solve_two_stage exactly.
flagged, info = flag_outliers(G, y, return_details=True)
x_new = refill_outliers(G, y, flagged)
print("suspected outliers:", np.flatnonzero(flagged))
for i in np.flatnonzero(flagged):
    print(f"  node {i}: y = {y[i]:.3f}, neighbours' median = {info['neighbour_median'][i]:.3f}, "
          f"z = {info['z'][i]:+.2f}  ->  new value {x_new[i]:.3f}")
assert np.allclose(x_new, solve_two_stage(G, y, k=2.5, lam=0.0, iterative="prune"))
print("same result as solve_two_stage: OK")

## 3. Final result on the real data

**Parameters: `k = 2.5`, `lam = 0`, `iterative = "prune"`** (the two-stage method of the group doc, with the victim-pruning pass).
- `k = 2.5`: the standard "moderately conservative" threshold of the MAD rule; with Gaussian noise it wrongly flags only about 1.2 % of clean nodes
  (about 0.4 of 32). See "The flagging rule and the role of `k`" in section 2.
- `lam = 0`: kept nodes keep their observed value; only flagged nodes are refilled.
- `iterative = "prune"`: re-judges the flagged nodes one at a time and un-flags "victims" (see "Improving Stage 1" in section 2, evidence in Appendix C).
- Checked in the appendices: Appendix A (`cv_score` on the real data) and Appendix B (synthetic data, seeds 0-19).
  They also report the variant with smoothing (`lam > 0`). Re-run the appendices with the group's real MSSV before relying on them.

In [ ]:
FINAL_PARAMS = {"k": 2.5, "lam": 0.0, "iterative": "prune"}   # main result of method 6 (justification in the cell above)

x_hat_real = solve_two_stage(G, y, **FINAL_PARAMS)
flagged_real, r_real, s_real = flag_nodes(G, y, k=FINAL_PARAMS["k"], iterative=FINAL_PARAMS["iterative"])
assert x_hat_real.shape == y.shape and np.all(np.isfinite(x_hat_real))

np.save("xhat_6.npy", x_hat_real)
print("flagged (suspected outlier) nodes on real data:", np.flatnonzero(flagged_real))
for i in np.flatnonzero(flagged_real):
    print(f"  node {i}: y = {y[i]:.3f} -> x_hat = {x_hat_real[i]:.3f}")

In [ ]:
# Figure: y vs x_hat on the graph; flagged nodes circled in red. Same colour scale for both panels.
vmin, vmax = min(y.min(), x_hat_real.min()), max(y.max(), x_hat_real.max())
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, vals, title in [(axes[0], y, "Observed y"), (axes[1], x_hat_real, "Recovered x_hat (method 6)")]:
    nodes = nx.draw_networkx_nodes(G, positions, node_color=vals, cmap="viridis",
                                   vmin=vmin, vmax=vmax, node_size=380, ax=ax)
    nx.draw_networkx_edges(G, positions, alpha=0.3, ax=ax)
    nx.draw_networkx_labels(G, positions, font_size=8, ax=ax)
    ax.set_title(title); ax.axis("off")
nx.draw_networkx_nodes(G, positions, nodelist=list(np.flatnonzero(flagged_real)), node_color="none",
                       edgecolors="red", linewidths=2.5, node_size=520, ax=axes[0])
fig.colorbar(nodes, ax=axes, label="value", shrink=0.8)
plt.savefig("fig_y_vs_xhat_method6.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Result row for the group comparison table

One row per method, as agreed in the group doc: method, parameters, synthetic RMSE (mean ± std over seeds 0-19), `cv_score` on the real data.
(How these two numbers are computed and how to read them: Appendix A and B.)

In [ ]:
synth_final = synth_rmse(FINAL_PARAMS)
result_row = {
    "method": "6. Two-stage: median/MAD outlier filter + harmonic inpainting",
    "params": FINAL_PARAMS,
    "synthetic_RMSE_mean": round(float(synth_final.mean()), 4),
    "synthetic_RMSE_std":  round(float(synth_final.std()), 4),
    "cv_score": round(float(cv_mean(FINAL_PARAMS)), 4),
}
print(result_row)

## 5. Write-up (Mục VI skeleton — edit in your own words, 1-1.5 pages)

**1. Assumptions.** The true signal `x` is smooth on the graph (neighbouring nodes have similar values). Observation = `x` + small Gaussian noise on all nodes + a *few* large outliers. Outliers are rare and not clustered, so most neighbours of any node are clean. Edge weights are a Gaussian kernel of the distance between sensors (heavier edge = closer, more similar sensors).

**2. Procedure / loss.**
Stage 1: `m_i` = weighted median of neighbours' `y`; `r_i = y_i - m_i`; `s = 1.4826·MAD(r)`; flag if `|r_i - med(r)| > k·s`; then re-judge the flagged nodes one at a time with the others removed from the medians, and un-flag those that no longer exceed the threshold ("victims" of neighbouring outliers).
Stage 2: with `F` flagged and `U` kept, `x_U = y_U` and `x_F = argmin x^T L x`, i.e. `L_FF x_F = -L_FU y_U`.
(Optional: `min Σ_U (x_i - y_i)^2 + lam x^T L x`.)

**3. Optimisation and parameters.** Closed form (one linear solve with `L_FF`, which is positive definite because the graph is connected and `U` is non-empty). `k = 2.5` from the robust-statistics literature (false positive rate ≈ 1.2 %), checked with 5-fold `cv_score` (Appendix A). Sanity check: flagged nodes on synthetic data should match the planted outliers (Appendix B).

**4. Experiments and limits.** Report the result row (section 4), the comparison tables of Appendix A and B, and the two figures. Limits: (a) the detector assumes most neighbours are clean, so adjacent outliers can hide each other; (b) with `lam=0` kept nodes keep their Gaussian noise, so the method cannot beat the noise floor on them; (c) a too-small `k` flags good nodes (loses information), a too-large `k` misses outliers; (d) with only a few nodes, MAD is itself noisy; (e) on 32 nodes the neighbour median uses few neighbours (about 5).

---
# Appendices (optional reading)

The main result above does not need them to be read. They contain the evidence behind the parameter choice:
- **Appendix A**: what `cv_score` is, and the grid search over `k` and `lam` on the real data.
- **Appendix B**: synthetic data with a known `x`: a sanity check on one seed and RMSE over seeds 0-19.
- **Appendix C**: the Stage 1 variants (original, prune, full iterative, floor on `s`) compared on all 100 public graphs.

Run them in order (B uses the variant found in A).

## Appendix A. Parameter check with `cv_score` (real data)

### What `cv_score` is
A **cross-validation score**: a way to judge parameters (`k`, `lam`) *without* the ground truth.
Hide some nodes, predict them from the rest, and check how close the predictions are to what those nodes actually read.
(The function comes from the shared code in the group doc, section 1.)

**How it works (5 folds)**
- **Step 1:** shuffle the 32 nodes and split them into 5 groups (folds) of 6-7 nodes.
- **Step 2:** for each fold in turn:
  - hide it (`mask = False` for those nodes): the method cannot see their `y`;
  - run the **whole method** on the visible nodes; it still returns an estimate for every node, hidden ones included;
  - record the error `|prediction − y|` for each hidden node.
- **Step 3:** after the 5 rounds every node has been hidden exactly once. `cv_score` = the **median** of all 32 errors. **Lower is better.**

Small example: hidden nodes 4, 9, 17 read `y = 19.1, 18.4, 20.9`; the method predicts 19.0, 18.9, 20.8; errors 0.1, 0.5, 0.1.
Repeat for all folds, then take the median of all errors.

**What happens inside one fold (how hidden nodes are predicted)**
Only 1/5 of the nodes is hidden at a time, so about 26 nodes stay visible. The full two-stage method runs on that fold:
- *Stage 1* uses only visible nodes: hidden nodes are not used as neighbours for `m_i`, and they are never flagged (they have no `y` to judge).
- *Stage 2* refills **hidden nodes and flagged nodes together**, both are simply "not trusted" (`keep = visible AND not flagged`).
  Each hidden node becomes the weighted average of its neighbours, exactly like a flagged node.
So outlier removal is not done once beforehand: it is redone inside every fold, on the visible nodes only.

Is that enough information? Every node has at least 4 neighbours. On the placeholder graph (5 different splits) every hidden node kept
at least 1 visible neighbour, usually 4. A hidden node with no visible neighbour would still get a value through the chain of its neighbours,
because the graph is connected.

**Why it is designed this way**
- No `x_true`: the hidden node's own reading `y` is the closest thing to the truth we have.
- **Median, not mean:** a hidden node may itself be an outlier, so its `y` is wrong and its error is huge even when the prediction is good. The median ignores those few cases.
- `cv_mean` averages `cv_score` over 5 random splits so one lucky or unlucky split does not decide the choice.

**Limits (for the write-up)**
- Predictions are compared with noisy `y`, not with `x`: even a perfect method scores above 0 (noise floor).
- It measures how well a method predicts *unseen* nodes, which is related to, but not the same as, the RMSE the graders compute on `x_true`.
- With only 32 nodes the score itself is noisy: small differences (e.g. 0.37 vs 0.39) may not mean much.

### Grid search
Grid over `k` (and `lam` for the optional extension), on the real `y`. Lower `cv_score` is better. `lam=0` is the pure method.
`k = 1e9` means "never flag", i.e. no outlier filter at all (a baseline).

In [ ]:
k_grid   = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 1e9]   # 1e9 = never flag => "no filter" baseline
lam_grid = [0.0, 0.05, 0.2, 1.0]

it = FINAL_PARAMS["iterative"]                              # same Stage 1 variant as the main result
table = {(k, lam): cv_mean({"k": k, "lam": lam, "iterative": it}) for k in k_grid for lam in lam_grid}
print("cv_score (rows k, cols lam)  lam:", lam_grid)
for k in k_grid:
    print(f"k={k:<7}", [round(float(table[(k, lam)]), 4) for lam in lam_grid])

best_pure = min(((k, 0.0) for k in k_grid), key=lambda p: table[p])
best_any  = min(table, key=table.get)
params_variant = {"k": best_any[0], "lam": best_any[1], "iterative": it}   # best smoothing variant, compared in Appendix B
print("chosen (main)     :", (FINAL_PARAMS["k"], FINAL_PARAMS["lam"]), "->", round(float(table[(FINAL_PARAMS["k"], FINAL_PARAMS["lam"])]), 4))
print("best pure (lam=0) :", best_pure, "->", round(float(table[best_pure]), 4))
print("best overall      :", best_any,  "->", round(float(table[best_any]), 4))

In [ ]:
# cv_score curve vs k (pure method, lam = 0)
ks = [k for k in k_grid if k < 1e8]
plt.figure(figsize=(6, 3.5))
plt.plot(ks, [table[(k, 0.0)] for k in ks], "o-", label="two-stage, lam=0")
plt.axhline(table[(1e9, 0.0)], color="gray", ls="--", label="no outlier filter")
plt.axvline(FINAL_PARAMS["k"], color="red", ls=":", label=f"chosen k = {FINAL_PARAMS['k']}")
plt.xlabel("threshold k"); plt.ylabel("cv_score (median |error|)"); plt.legend(); plt.grid(alpha=.3)
plt.title("Checking k by cross-validation")
plt.savefig("fig_cv_k_method6.png", dpi=150, bbox_inches="tight"); plt.show()

**How to read the grid** (check again after setting the real MSSV):
- Compare the chosen `k` with the "never flag" row: the difference is what the outlier filter itself contributes.
- In the `lam > 0` columns, if the rows for large `k` (no flagging) score about the same as the best row, the gain comes from smoothing, not from the filter.
- Differences of a few thousandths are within the noise of a 32-node CV.

## Appendix B. Synthetic data (known `x`)

`make_synthetic` (shared code, section 1) builds a smooth `x` on our graph, adds Gaussian noise and 3 outliers, so RMSE against the true `x` can be measured.
This only reflects the generator's assumptions, not necessarily the real data.

### B.1 Sanity check on one seed
Are the planted outliers (largest `|y_syn − x_true|`) among the flagged nodes?

In [ ]:
x_true, y_syn = make_synthetic(G, y, seed=0)
flagged, r, s = flag_nodes(G, y_syn, k=FINAL_PARAMS["k"], iterative=FINAL_PARAMS["iterative"])
x_hat = solve_two_stage(G, y_syn, **FINAL_PARAMS)
print("flagged nodes:", np.flatnonzero(flagged))
print("planted outliers (largest |y_syn - x_true|):", np.argsort(-np.abs(y_syn - x_true))[:3])
print("RMSE of y itself  :", round(rmse(y_syn, x_true), 4))
print("RMSE of two-stage :", round(rmse(x_hat, x_true), 4))

### B.2 RMSE over seeds 0-19
Baseline (`x_hat = y`, no processing), the main parameters, and the best variant from Appendix A.

In [ ]:
base = np.array([rmse(make_synthetic(G, y, sd)[1], make_synthetic(G, y, sd)[0]) for sd in range(20)])
r_main, r_var = synth_rmse(FINAL_PARAMS), synth_rmse(params_variant)

print(f"baseline (x_hat = y)              : {base.mean():.4f} ± {base.std():.4f}")
print(f"main    {FINAL_PARAMS}: {r_main.mean():.4f} ± {r_main.std():.4f}")
print(f"variant {params_variant}: {r_var.mean():.4f} ± {r_var.std():.4f}")

## Appendix C. Stage 1 variants on all 100 public graphs

Reproduces the table in section 2 (Stage 1, "Improving Stage 1"). Every graph in `public_cases.json` is used:
- **synthetic**: 5 seeds per graph (500 runs), generator identical to `make_synthetic` but on each graph; we know which 3 nodes are planted outliers;
- **real data**: flags on each case's real `y`, and `cv_score` averaged over 5 splits.

Takes about half a minute.

In [ ]:
with open("public_cases.json", encoding="utf-8") as f:
    ALL = json.load(f)["cases"]

def load_case(cid):
    c = ALL[cid]; Gc = nx.Graph()
    for i, xy in enumerate(c["positions"]): Gc.add_node(i, pos=tuple(map(float, xy)))
    for u, v, w in c["edges"]: Gc.add_edge(int(u), int(v), weight=float(w))
    return Gc, np.asarray(c["y"], dtype=float)

def synthetic_on(Gc, yc, seed):
    # Same recipe as make_synthetic (shared code), but for any graph; also returns the planted outlier nodes.
    nc = len(yc); Lc = nx.laplacian_matrix(Gc, nodelist=range(nc), weight="weight").toarray()
    rng = np.random.default_rng(seed); lam_, U = np.linalg.eigh(Lc)
    x = U[:, :5] @ (rng.normal(size=5) / (1 + lam_[:5])); x = np.median(yc) + np.std(yc) * x / np.std(x)
    sigma = 0.1 * np.std(yc); ys = x + rng.normal(0, sigma, nc)
    idx = rng.choice(nc, 3, replace=False); ys[idx] += rng.choice([-1, 1], 3) * rng.uniform(4, 8, 3) * sigma
    return x, ys, set(idx.tolist())

def cv_on(Gc, yc, params, seeds=range(5), k=5):
    out = []
    for sd in seeds:
        folds = np.random.default_rng(sd).permutation(len(yc)) % k; err = []
        for f in range(k):
            mk = folds != f; xh = solve_two_stage(Gc, yc, mask=mk, **params); err.append(np.abs(xh[~mk] - yc[~mk]))
        out.append(np.median(np.concatenate(err)))
    return np.mean(out)

VARIANTS = {"original": {"iterative": False}, "prune": {"iterative": "prune"}, "full iterative": {"iterative": True},
            "floor on s": {"iterative": False, "s_floor": 0.49}, "full iterative + floor": {"iterative": True, "s_floor": 0.49}}
case_ids = sorted(ALL, key=lambda x: (0, int(x)) if x.isdecimal() else (1, x))
graphs = {cid: load_case(cid) for cid in case_ids}

print(f"{'variant':24s} {'synth RMSE':>10s} {'caught':>7s} {'false/run':>9s} {'adj-flag cases':>14s} {'mean cv':>8s}")
for name, opt in VARIANTS.items():
    p = {"k": 2.5, "lam": 0.0, **opt}
    rm, hit, false, adj, cvs = [], 0, 0, 0, []
    for cid, (Gc, yc) in graphs.items():
        for sd in range(5):
            x, ys, planted = synthetic_on(Gc, yc, sd)
            F = set(np.flatnonzero(flag_nodes(Gc, ys, k=2.5, iterative=p.get("iterative", False), s_floor=p.get("s_floor", 0.0))[0]).tolist())
            rm.append(rmse(solve_two_stage(Gc, ys, **p), x)); hit += len(F & planted); false += len(F - planted)
        Fr = np.flatnonzero(flag_nodes(Gc, yc, k=2.5, iterative=p.get("iterative", False), s_floor=p.get("s_floor", 0.0))[0]).tolist()
        adj += any(len(cc) > 1 for cc in nx.connected_components(Gc.subgraph(Fr)))
        cvs.append(cv_on(Gc, yc, p))
    runs = 5 * len(graphs)
    print(f"{name:24s} {np.mean(rm):10.4f} {100*hit/(3*runs):6.0f}% {false/runs:9.2f} {adj:14d} {np.mean(cvs):8.4f}")